In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device('mps')
    print(f'Using device: MPS (Apple Silicon GPU)')
elif torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'Using device: CUDA (NVIDIA GPU)')
else:
    device = torch.device('cpu')
    print(f'Using device: CPU')
print(f'PyTorch version: {torch.__version__}')

In [ ]:
# Loading train data
df_train = pd.read_csv('mitbih_train.csv', header=None)
print(f"Train data loaded: {df_train.shape}")
# Loading test data
df_test = pd.read_csv('mitbih_test.csv', header=None)
print(f"Test data loaded: {df_test.shape}")

In [ ]:
# Separate features (X) and labels (y)
X_train = df_train.iloc[:, :-1].values
y_train = df_train.iloc[:, -1].values.astype(int)
X_test = df_test.iloc[:, :-1].values
y_test = df_test.iloc[:, -1].values.astype(int)

In [ ]:
print("\n=== Class Distribution in Training Set ===")
class_names = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unclassifiable']
for i in range(5):
    count = np.sum(y_train == i)
    percentage = 100 * count / len(y_train)
    print(f"Class {i} ({class_names[i]}): {count} ({percentage:.2f}%)")
print("\n=== Class Distribution in Test Set ===")
for i in range(5):
    count = np.sum(y_test == i)
    percentage = 100 * count / len(y_test)
    print(f"Class {i} ({class_names[i]}): {count} ({percentage:.2f}%)")

In [ ]:
print("\n=== Data Quality Analysis ===")
print("\n1. Missing Values Check:")
print(f"Training set - Missing values: {df_train.isnull().sum().sum()}")
print(f"Test set - Missing values: {df_test.isnull().sum().sum()}")
if df_train.isnull().sum().sum() > 0:
    print("WARNING: Found missing values in training data!")
    print(df_train.isnull().sum()[df_train.isnull().sum() > 0])
if df_test.isnull().sum().sum() > 0:
    print("WARNING: Found missing values in test data!")
    print(df_test.isnull().sum()[df_test.isnull().sum() > 0])

In [ ]:
print("\n2. Duplicate Records Check:")
train_duplicates = df_train.duplicated().sum()
test_duplicates = df_test.duplicated().sum()
print(f"Training set - Duplicate rows: {train_duplicates} ({100*train_duplicates/len(df_train):.2f}%)")
print(f"Test set - Duplicate rows: {test_duplicates} ({100*test_duplicates/len(df_test):.2f}%)")

In [ ]:
if train_duplicates > 0:
    print(f"\nRemoving {train_duplicates} duplicate rows from training set...")
    df_train = df_train.drop_duplicates()
    X_train = df_train.iloc[:, :-1].values
    y_train = df_train.iloc[:, -1].values.astype(int)
    print(f"New training set shape: {df_train.shape}")
if test_duplicates > 0:
    print(f"\nRemoving {test_duplicates} duplicate rows from test set...")
    df_test = df_test.drop_duplicates()
    X_test = df_test.iloc[:, :-1].values
    y_test = df_test.iloc[:, -1].values.astype(int)
    print(f"New test set shape: {df_test.shape}")

In [ ]:
print("\n3. Infinite Values Check:")
train_inf = np.isinf(X_train).sum()
test_inf = np.isinf(X_test).sum()
print(f"Training set - Infinite values: {train_inf}")
print(f"Test set - Infinite values: {test_inf}")
if train_inf > 0 or test_inf > 0:
    print("WARNING: Found infinite values in data!")

In [ ]:
print("\n4. Data Range Analysis:")
print(f"Training set - Min: {X_train.min():.4f}, Max: {X_train.max():.4f}, Mean: {X_train.mean():.4f}, Std: {X_train.std():.4f}")
print(f"Test set - Min: {X_test.min():.4f}, Max: {X_test.max():.4f}, Mean: {X_test.mean():.4f}, Std: {X_test.std():.4f}")

In [ ]:
print("\n5. Label Validation:")
unique_train_labels = np.unique(y_train)
unique_test_labels = np.unique(y_test)
print(f"Training set - Unique labels: {unique_train_labels}")
print(f"Test set - Unique labels: {unique_test_labels}")
if not np.array_equal(unique_train_labels, np.array([0, 1, 2, 3, 4])):
    print("WARNING: Unexpected labels found in training set!")
if not np.array_equal(unique_test_labels, np.array([0, 1, 2, 3, 4])):
    print("WARNING: Unexpected labels found in test set!")

In [ ]:
print("\n=== Class Distribution (Imbalance Analysis) ===")
class_names = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion', 'Unclassifiable']
train_counts = [np.sum(y_train == i) for i in range(5)]
test_counts = [np.sum(y_test == i) for i in range(5)]
print("\nTraining Set:")
for i in range(5):
    count = train_counts[i]
    percentage = 100 * count / len(y_train)
    print(f"  Class {i} ({class_names[i]:20s}): {count:6d} samples ({percentage:5.2f}%)")
print("\nTest Set:")
for i in range(5):
    count = test_counts[i]
    percentage = 100 * count / len(y_test)
    print(f"  Class {i} ({class_names[i]:20s}): {count:6d} samples ({percentage:5.2f}%)")

In [ ]:
max_class_count = max(train_counts)
min_class_count = min(train_counts)
imbalance_ratio = max_class_count / min_class_count
print(f"\nImbalance Ratio: {imbalance_ratio:.2f}:1")
if imbalance_ratio > 10:
    print("Dataset is HIGHLY IMBALANCED - Class weighting will be applied")

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(range(5), train_counts, color='steelblue', edgecolor='black', alpha=0.8)
plt.xlabel('Class', fontsize=12)
plt.ylabel('Number of Samples', fontsize=12)
plt.title('Class Distribution - Training Set', fontsize=14, fontweight='bold')
plt.xticks(range(5), [f'Class {i}\n{class_names[i]}' for i in range(5)])
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import plotly.express as px
labels = {
    0: "Normal",
    1: "Supraventricular Premature",
    2: "Premature Ventricular Contraction",
    3: "Fusion of Ventricular and Normal",
    4: "Unclassifiable Beat"
}
value_counts = df_train.iloc[:, -1].value_counts().sort_index().rename(labels)
bar_fig = px.bar(x=value_counts.index, y=value_counts.values,
                labels={'x': 'Heartbeat Type', 'y': 'Number of Samples'},
                text_auto=True,
                title="Distribution of Heartbeat Types in Training Dataset",
                color=value_counts.values,
                color_continuous_scale='Blues')
pie_fig = px.pie(names=value_counts.index, values=value_counts.values,
                title="Percentage Distribution of Heartbeat Types in Training Dataset",
                hole=0.3)

bar_fig.update_layout(title_x=0.5, width=900, height=600, showlegend=False)
pie_fig.update_layout(title_x=0.5, width=900, height=600)

bar_fig.show()
pie_fig.show()

In [ ]:
test_counts = [np.sum(y_test == i) for i in range(5)]
ax2.bar(range(5), test_counts, color='coral', edgecolor='black')
ax2.set_xlabel('Class')
ax2.set_ylabel('Number of Samples')
ax2.set_title('Class Distribution - Test Set')
ax2.set_xticks(range(5))
ax2.set_xticklabels([f'{i}\n{class_names[i]}' for i in range(5)], rotation=45, ha='right')
ax2.grid(axis='y', alpha=0.3)

In [ ]:
for i, count in enumerate(test_counts):
    percentage = 100 * count / len(y_test)
    ax2.text(i, count, f'{percentage:.1f}%', ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(15, 10))
for i in range(5):
    idx = np.where(y_train == i)[0][0]
    axes[i].plot(X_train[idx])
    axes[i].set_title(f'Class {i}: {class_names[i]}')
    axes[i].set_xlabel('Time')
    axes[i].set_ylabel('Amplitude')
    axes[i].grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print("\n=== Sample ECG Signals Visualization ===")
plt.figure(figsize=(15, 7))
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E']
for i in range(5):
    class_indices = np.where(y_train == i)[0]
    idx = class_indices[5] if len(class_indices) > 5 else class_indices[0]
    plt.plot(X_train[idx], linewidth=2, color=colors[i],
            label=f'Class {i}: {class_names[i]}', alpha=0.8)
plt.title('ECG Signal Comparison Across All Classes', fontsize=14, fontweight='bold')
plt.xlabel('Time Steps', fontsize=12)
plt.ylabel('Amplitude', fontsize=12)
plt.legend(loc='upper right', fontsize=10, framealpha=0.9)
plt.grid(True, alpha=0.3)
plt.xlim(0, 186)
plt.tight_layout()
plt.show()
plt.tight_layout()
plt.show()
print("\n" + "="*70)
print("Data quality checks and class distribution analysis complete!")
print("="*70)

In [ ]:
print("\n=== Single ECG Signal Example (Row 5) ===")
sample_idx = 5
sample_signal = X_train[sample_idx]
sample_label = y_train[sample_idx]
plt.figure(figsize=(15, 5))
plt.plot(sample_signal, linewidth=2, color='crimson')
plt.title(f'ECG Signal - Row {sample_idx} | Class: {sample_label} ({class_names[sample_label]})', 
        fontsize=14, fontweight='bold')
plt.xlabel('Time Steps', fontsize=12)
plt.ylabel('Amplitude', fontsize=12)
plt.grid(True, alpha=0.3)
plt.xlim(0, 186)
plt.tight_layout()
plt.show()
print(f"Signal from row {sample_idx} belongs to Class {sample_label}: {class_names[sample_label]}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
class ECGDataset(Dataset):
    """Custom PyTorch Dataset for ECG signals"""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X).unsqueeze(1)
        self.y = torch.LongTensor(y)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
train_dataset = ECGDataset(X_train_scaled, y_train)
test_dataset = ECGDataset(X_test_scaled, y_test)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = batch_size, shuffle = False)

In [ ]:
class CNN_LSTM(nn.Module):
    """
    Hybrid CNN-LSTM architecture for ECG signal classification
    
    CNN layers: Extract spatial features from signals
    LSTM layers: Capture temporal dependencies
    """
    def __init__(self, n_classes=5, input_channels=1, input_length=187):
        super(CNN_LSTM, self).__init__()
        self.conv1 = nn.Conv1d(input_channels, 32, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)
        self.dropout1 = nn.Dropout(0.2)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)
        self.dropout2 = nn.Dropout(0.2)
        self.conv3 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(2)
        self.dropout3 = nn.Dropout(0.2)
        lstm_input_size = input_length // 8
        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3,
            bidirectional=True
        )
        self.fc1 = nn.Linear(128, 64)
        self.dropout4 = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, n_classes)
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.pool1(x)
        x = self.dropout1(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.pool2(x)
        x = self.dropout2(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x = F.relu(x)
        x = self.pool3(x)
        x = self.dropout3(x)
        x = x.permute(0, 2, 1)
        lstm_out, (hidden, cell) = self.lstm(x)
        x = lstm_out[:, -1, :]
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout4(x)
        x = self.fc2(x)
        return x

model = CNN_LSTM(n_classes=5, input_channels=1, input_length=187)
model = model.to(device)
print(model)
print(f'\nTotal number of parameters: {sum(p.numel() for p in model.parameters())}')

In [ ]:
class_counts = np.bincount(y_train)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print(f"\nClass weights for loss function:")
for i, weight in enumerate(class_weights):
    print(f"  Class {i} ({class_names[i]}): {weight:.4f}")

In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5
)

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train model for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
    """Validate model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    val_loss = running_loss / len(dataloader)
    val_acc = 100 * correct / total
    return val_loss, val_acc

In [ ]:
num_epochs = 50
train_losses, train_accs = [], []
val_losses, val_accs = [], []
best_val_acc = 0.0
print("\nStarting training...\n")
print(f"{'Epoch':<8} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12}")
print("="*60)
for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    scheduler.step(val_loss)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_ecg_model.pth')
        print(f'{epoch+1:<8} {train_loss:<12.4f} {train_acc:<12.2f}% {val_loss:<12.4f} {val_acc:<12.2f}% ⭐ BEST')
    else:
        print(f'{epoch+1:<8} {train_loss:<12.4f} {train_acc:<12.2f}% {val_loss:<12.4f} {val_acc:<12.2f}%')
print("\n" + "="*60)
print(f'Training complete! Best validation accuracy: {best_val_acc:.2f}%')
print("="*60)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses, label='Validation Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)
ax2.plot(train_accs, label='Train Accuracy')
ax2.plot(val_accs, label='Validation Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*70)
print("FINAL MODEL EVALUATION")
print("="*70)
model.load_state_dict(torch.load('best_ecg_model.pth'))
model.eval()
y_true = []
y_pred = []
y_probs = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probabilities = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        y_probs.extend(probabilities.cpu().numpy())

In [ ]:
print("\n" + "="*70)
print("FINAL MODEL EVALUATION")
print("="*70)
model.load_state_dict(torch.load('best_ecg_model.pth'))
model.eval()
y_true = []
y_pred = []
y_probs = []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        probabilities = F.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        y_probs.extend(probabilities.cpu().numpy())
y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_probs = np.array(y_probs)
overall_acc = accuracy_score(y_true, y_pred) * 100
print(f'\n📊 OVERALL TEST ACCURACY: {overall_acc:.2f}%\n')
print("="*70)
print("PER-CLASS PERFORMANCE")
print("="*70)
for i in range(5):
    class_mask = y_true == i
    class_acc = accuracy_score(y_true[class_mask], y_pred[class_mask]) * 100
    class_total = np.sum(class_mask)
    class_correct = np.sum((y_true[class_mask] == y_pred[class_mask]))
    print(f"{class_names[i]:30s}: {class_correct:4d}/{class_total:4d} correct ({class_acc:6.2f}%)")
print("\n" + "="*70)
print("CONFUSION MATRIX")
print("="*70)
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))
from sklearn.metrics import precision_recall_fscore_support, cohen_kappa_score
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average='weighted')
kappa = cohen_kappa_score(y_true, y_pred)
print("\n" + "="*70)
print("AGGREGATE METRICS")
print("="*70)
print(f"Weighted Precision: {precision*100:.2f}%")
print(f"Weighted Recall:    {recall*100:.2f}%")
print(f"Weighted F1-Score:  {f1*100:.2f}%")
print(f"Cohen's Kappa:      {kappa:.4f}")

In [ ]:
print(f"y_probs shape: {y_probs.shape}")
print(f"y_probs type: {type(y_probs)}")
print(f"y_probs sample (first 3 rows):")
print(y_probs[:3])
print(f"\ny_true_bin shape: {y_true_bin.shape}")
print(f"y_true_bin sample (first 3 rows):")
print(y_true_bin[:3])

In [ ]:
print("\n" + "="*70)
print("ROC CURVES - MULTICLASS")
print("="*70)
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3, 4])
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E']
for i in range(5):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax1.plot(fpr, tpr, color=colors[i], lw=3, alpha=0.8,
            label=f'{class_names[i]} (AUC = {roc_auc:.4f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC = 0.5000)')
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('False Positive Rate', fontsize=12)
ax1.set_ylabel('True Positive Rate', fontsize=12)
ax1.set_title('ROC Curves - Full View', fontsize=13, fontweight='bold')
ax1.legend(loc="lower right", fontsize=9)
ax1.grid(alpha=0.3)
for i in range(5):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    ax2.plot(fpr, tpr, color=colors[i], lw=3, alpha=0.8,
            label=f'{class_names[i]} (AUC = {roc_auc:.4f})')
ax2.plot([0, 0.2], [0.8, 1.0], 'k--', lw=2, alpha=0.3)
ax2.set_xlim([0.0, 0.2])
ax2.set_ylim([0.8, 1.0])
ax2.set_xlabel('False Positive Rate', fontsize=12)
ax2.set_ylabel('True Positive Rate', fontsize=12)
ax2.set_title('ROC Curves - Zoomed View', fontsize=13, fontweight='bold')
ax2.legend(loc="lower right", fontsize=9)
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("\nArea Under Curve (AUC) per class:")
for i in range(5):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    print(f"  {class_names[i]:30s}: {roc_auc:.4f}")

In [ ]:
from sklearn.metrics import roc_auc_score
macro_auc = roc_auc_score(y_true_bin, y_probs, average='macro')
micro_auc = roc_auc_score(y_true_bin, y_probs, average='micro')
print(f"\n  {'Macro-average AUC':30s}: {macro_auc:.4f}")
print(f"  {'Micro-average AUC':30s}: {micro_auc:.4f}")
print("\n" + "="*70)
print("MODEL EVALUATION COMPLETE")
print("="*70)

In [ ]:
def predict_ecg(model, signal, scaler, device):
    """
    Prediction on a single ECG signal
    Args:
        model: Trained PyTorch model
        signal: NumPy array of ECG signal (1D)
        scaler: StandardScaler used for normalization
        device: CPU or GPU
    Returns:
        predicted_class, probabilities
    """
    model.eval()
    signal_scaled = scaler.transform(signal.reshape(1, -1))
    signal_tensor = torch.FloatTensor(signal_scaled).unsqueeze(1).to(device)
    with torch.no_grad():
        output = model(signal_tensor)
        probabilities = F.softmax(output, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
    return predicted_class, probabilities.cpu().numpy()[0]

In [ ]:
sample_signal = X_test[0]
pred_class, probs = predict_ecg(model, sample_signal, scaler, device)
print(f"\nExample prediction:")
print(f"Predicted class: {pred_class} ({class_names[pred_class]})")
print(f"Probabilities:")
for i, prob in enumerate(probs):
    print(f"  {class_names[i]}: {prob*100:.2f}%")
print(f"True class: {y_test[0]} ({class_names[y_test[0]]})")

In [ ]:
print("\n" + "="*70)
print("EXAMPLE PREDICTION WITH VISUALIZATION")
print("="*70)
sample_idx = 0
sample_signal = X_test[sample_idx]
true_label = y_test[sample_idx]
pred_class, probs = predict_ecg(model, sample_signal, scaler, device)
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))
ax1.plot(sample_signal, linewidth=2, color='darkblue')
ax1.set_title(f'ECG Signal Sample #{sample_idx}', fontsize=14, fontweight='bold')
ax1.set_xlabel('Time Steps', fontsize=12)
ax1.set_ylabel('Amplitude', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 186)
textstr = f'True Class: {true_label} ({class_names[true_label]})\nPredicted: {pred_class} ({class_names[pred_class]})'
props = dict(boxstyle='round', facecolor='wheat' if pred_class == true_label else 'lightcoral', alpha=0.8)
ax1.text(0.02, 0.98, textstr, transform=ax1.transAxes, fontsize=11,
        verticalalignment='top', bbox=props)
colors_bar = ['green' if i == pred_class else 'steelblue' for i in range(5)]
bars = ax2.bar(range(5), probs * 100, color=colors_bar, edgecolor='black', alpha=0.7)
ax2.set_xlabel('Class', fontsize=12)
ax2.set_ylabel('Probability (%)', fontsize=12)
ax2.set_title('Model Confidence for Each Class', fontsize=14, fontweight='bold')
ax2.set_xticks(range(5))
ax2.set_xticklabels([f'{i}\n{class_names[i]}' for i in range(5)], rotation=0)
ax2.set_ylim(0, 105)
ax2.grid(axis='y', alpha=0.3)
for i, (bar, prob) in enumerate(zip(bars, probs)):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{prob*100:.2f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()
print(f"\n{'='*70}")
print(f"Sample Index: {sample_idx}")
print(f"True Class: {true_label} ({class_names[true_label]})")
print(f"Predicted Class: {pred_class} ({class_names[pred_class]})")
print(f"Prediction Status: {'✅ CORRECT' if pred_class == true_label else '❌ INCORRECT'}")
print(f"\nProbability Distribution:")
for i, prob in enumerate(probs):
    marker = "👉" if i == pred_class else "  "
    print(f"{marker} {class_names[i]:30s}: {prob*100:6.2f}%")
print(f"{'='*70}")